In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [ ]:
pip install --upgrade transformers

In [5]:
pip install numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 104.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# Load Jigsaw dataset
jigsaw_dir = Path("/content/drive/My Drive/Jigsaw")
df = pd.read_csv(jigsaw_dir /'train.csv')

# Define binary toxicity label (toxic if any of the categories are 1)
df['toxic_label'] = (df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].sum(axis=1) > 0).astype(int)
df = df[['comment_text', 'toxic_label']]
df = df.rename(columns={"comment_text": "text"})
# Split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for Trainer
train_dataset = train_dataset.rename_column("toxic_label", "label")
test_dataset = test_dataset.rename_column("toxic_label", "label")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta-toxic-jigsaw",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()
model_dir = Path("/content/drive/My Drive/DALI/model")
# Save model
model.save_pretrained(model_dir / "roberta-toxic-jigsaw")
tokenizer.save_pretrained(model_dir / "roberta-toxic-jigsaw")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3-678652878.py:70: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.099200,0.095569,0.970422,0.850349


Evaluation: {'eval_loss': 0.09556921571493149, 'eval_accuracy': 0.9704223586915653, 'eval_f1': 0.8503487634749525, 'eval_runtime': 104.4349, 'eval_samples_per_second': 152.803, 'eval_steps_per_second': 9.556, 'epoch': 1.0}


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

model_dir = Path("/content/drive/My Drive/DALI/model")

# Lyrics
lyrics_dir = Path("/content/drive/My Drive/DALI/unzipped_files/lyricsline/all_lyrics_labeled.csv")
lyrics_df = pd.read_csv(lyrics_dir)
lyrics_df = lyrics_df.dropna(subset=['text'])


# Define binary toxicity label (toxic if any of the categories are 1)
# Split
train_df, test_df = train_test_split(
    lyrics_df,
    test_size=0.1,
    random_state=42,
    stratify=lyrics_df["label"]  # Ensures label distribution is preserved
)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = str(model_dir / "roberta-toxic-jigsaw")

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

# Set format for Trainer
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta-toxic-jigsaw",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save model
model.save_pretrained(model_dir/"roberta-lyrics")
tokenizer.save_pretrained(model_dir/"roberta-lyrics")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)


Map:   0%|          | 0/78178 [00:00<?, ? examples/s]

Map:   0%|          | 0/8687 [00:00<?, ? examples/s]

/tmp/ipython-input-27-3859015595.py:75: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.017800,0.014753,0.997237,0.939698
2,0.007900,0.007287,0.998849,0.974359
3,0.002700,0.009835,0.998388,0.964467


Evaluation: {'eval_loss': 0.007286811247467995, 'eval_accuracy': 0.9988488546103372, 'eval_f1': 0.9743589743589743, 'eval_runtime': 54.8626, 'eval_samples_per_second': 158.341, 'eval_steps_per_second': 9.897, 'epoch': 3.0}


In [2]:
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

model_dir = Path("/content/drive/My Drive/DALI/model")

# Lyrics
lyrics_dir = Path('/content/drive/MyDrive/DALI/clean_explicit_comparison/lyrics_labeled.csv')
lyrics_df = pd.read_csv(lyrics_dir)
lyrics_df = lyrics_df.dropna(subset=['text'])


# Define binary toxicity label (toxic if any of the categories are 1)
# Split
train_df, test_df = train_test_split(
    lyrics_df,
    test_size=0.1,
    random_state=42,
    stratify=lyrics_df["label"]  # Ensures label distribution is preserved
)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = Path("/content/drive/My Drive/DALI/model/roberta-lyrics")

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

# Set format for Trainer
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta-toxic-jigsaw",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save model
model.save_pretrained(model_dir/"roberta-lyrics_final")
tokenizer.save_pretrained(model_dir/"roberta-lyrics_final")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)


Map:   0%|          | 0/1270 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

/tmp/ipython-input-2-3269852313.py:75: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.318190,0.887324,0.741935
2,No log,0.336217,0.894366,0.761905
3,No log,0.339135,0.901408,0.781250


Evaluation: {'eval_loss': 0.31818950176239014, 'eval_accuracy': 0.8873239436619719, 'eval_f1': 0.7419354838709677, 'eval_runtime': 0.9966, 'eval_samples_per_second': 142.483, 'eval_steps_per_second': 9.031, 'epoch': 3.0}
